<a href="https://colab.research.google.com/github/douglasldd/-blog-simples-usando-django-de-acordo-com-o-tutorial-django-girls/blob/main/Exercicio13AGO26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Bibliotecas principais do Módulo 4 import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

from pathlib import Path

from datetime import datetime
import unicodedata
import pandas as pd

pd.set_option("display.max_columns", 120)

pd.set_option("display.width", 160)

In [ ]:
# Estrutura padrão do projeto
RAIZ = Path(".")
PASTAS = [
"dados_brutos", "dados_tratados",
"notebooks", "sql", "Douglas", "dashboards", "relatorios", "apresentacao", "logs"]

for pasta in PASTAS:
  (RAIZ / pasta).mkdir(parents=True, exist_ok=True)
  print("Pastas verificadas/criadas:")

for pasta in PASTAS:
  print("-", RAIZ / pasta)

Pastas verificadas/criadas:
Pastas verificadas/criadas:
Pastas verificadas/criadas:
Pastas verificadas/criadas:
Pastas verificadas/criadas:
Pastas verificadas/criadas:
Pastas verificadas/criadas:
Pastas verificadas/criadas:
Pastas verificadas/criadas:
- dados_brutos
- dados_tratados
- notebooks
- sql
- Douglas
- dashboards
- relatorios
- apresentacao
- logs


In [ ]:
ARQUIVO_BRUTO = Path("dados_brutos/acidentes2025.csv")
ARQUIVO_BASE_ANALITICA = Path("dados_tratados/base_analitica_prf_2025.csv")
ARQUIVO_BASE_MODELAVEL = Path("dados_tratados/base_modelavel_prf_2025.csv")
ARQUIVO_DICIONARIO = Path("dados_tratados/dicionario_variaveis_modulo4.csv")
ARQUIVO_DECISOES = Path("logs/decisoes_tratamento_modulo4.md")
ARQUIVO_README = Path("README.md")

SEPARADOR = ";"
ENCODING_ENTRADA = "latin1"
ENCODING_SAIDA = "utf-8-sig"

In [ ]:
def ler_csv_prf(caminho, sep=";", encodings=("latin1","utf-8","utf-8-sig")):
    ultimo_erro = None
    for enc in encodings:
        try:
            print(f"Tentando encoding={enc}...")
            return pd.read_csv(
                caminho, sep=sep,
                encoding=enc, low_memory=False)
        except Exception as erro:
            ultimo_erro = erro
            print(f"Falhou com {enc}: {erro}")
    raise ultimo_erro

df = ler_csv_prf(ARQUIVO_BRUTO, sep=SEPARADOR)
df.head()

Tentando encoding=latin1...


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
0,652493,01/01/2025,quarta-feira,06:20:00,SP,116,225,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP
1,652519,01/01/2025,quarta-feira,07:50:00,CE,116,"546,2",PENAFORTE,Pista esburacada,Colisão frontal,NaN,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE
2,652522,01/01/2025,quarta-feira,08:45:00,PR,369,"88,2",CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,5,0,3,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR
3,652544,01/01/2025,quarta-feira,11:00:00,PR,116,74,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR
4,652549,01/01/2025,quarta-feira,09:30:00,MG,251,471,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG


In [ ]:
def normalizar_nome_coluna(nome):
    nome = str(nome).strip().lower()
    nome = unicodedata.normalize(
        "NFKD", nome
    ).encode("ascii","ignore").decode("utf-8")
    nome = nome.replace(" ","_")\
        .replace("-","_").replace("/","_")
    while " " in nome:
        nome = nome.replace(" ","_")
    return nome.strip("_")

df.columns = [normalizar_nome_coluna(c)
for c in df.columns]
renomear = { "condicao_meteorologica": "condicao_metereologica"}
df = df.rename(columns={
k:v for k,v in renomear.items() if k in df.columns})

In [ ]:
colunas_esperadas = [
    "data_inversa", "dia_semana", "horario", "uf", "br", "municipio",
    "causa_acidente", "tipo_acidente", "classificacao_acidente", "fase_dia",
    "condicao_meteorologica", "tipo_pista", "tracado_via", "uso_solo",
    "pessoas", "mortos", "feridos_leves", "feridos_graves", "feridos", "veiculos"
]

# Procura quais colunas esperadas não existem no seu DataFrame
faltantes = [c for c in colunas_esperadas if c not in df.columns]
print("Colunas faltantes:", faltantes)

# Adicionado o recuo correto (quatro espaços) nas linhas de dentro do if
if faltantes:
    print("Atenção: ajuste nomes ou confirme o dicionário da PRF.")


Colunas faltantes: ['condicao_meteorologica']
Atenção: ajuste nomes ou confirme o dicionário da PRF.


In [ ]:
df.head()

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
0,652493,01/01/2025,quarta-feira,06:20:00,SP,116,225,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP
1,652519,01/01/2025,quarta-feira,07:50:00,CE,116,"546,2",PENAFORTE,Pista esburacada,Colisão frontal,NaN,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE
2,652522,01/01/2025,quarta-feira,08:45:00,PR,369,"88,2",CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,5,0,3,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR
3,652544,01/01/2025,quarta-feira,11:00:00,PR,116,74,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR
4,652549,01/01/2025,quarta-feira,09:30:00,MG,251,471,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG


In [ ]:
 #AMOSTRA ALEATORIA
 df.sample(5)

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
7120,695192,01/06/2025,domingo,18:45:00,MG,40,"746,3",SANTOS DUMONT,Velocidade Incompatível,Saída de leito carroçável,Com Vítimas Feridas,Plena Noite,Crescente,Nublado,Simples,Curva,Não,2,0,1,0,1,0,1,2,"-21,472","-43,552",SPRF-MG,DEL05-MG,UOP01-DEL05-MG
47844,706975,24/07/2025,quinta-feira,17:50:00,RO,319,63,PORTO VELHO,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Plena Noite,Crescente,Ignorado,Múltipla,Reta;Interseção de Vias,Sim,1,0,0,1,0,0,1,1,"-8,75662817","-63,88658937",SPRF-RO,DEL01-RO,UOP01-DEL01-RO
63625,730537,08/11/2025,sábado,19:30:00,RS,116,149,CAXIAS DO SUL,Condutor desrespeitou a iluminação vermelha do...,Colisão transversal,Com Vítimas Feridas,Plena Noite,Crescente,Céu Claro,Múltipla,Interseção de Vias,Sim,3,0,2,0,1,0,2,2,"-29,1604276","-51,162471",SPRF-RS,DEL05-RS,UOP01-DEL05-RS
59774,724702,12/10/2025,domingo,23:30:00,RJ,40,"68,3",PETROPOLIS,Problema com o freio,Colisão com objeto,Com Vítimas Feridas,Plena Noite,Decrescente,Nublado,Simples,Curva,Não,3,0,1,0,2,0,1,1,"-22,44184642","-43,18834869",SPRF-RJ,DEL05-RJ,UOP02-DEL05-RJ
52786,714324,27/08/2025,quarta-feira,17:30:00,PB,230,155,MASSARANDUBA,Acumulo de areia ou detritos sobre o pavimento,Saída de leito carroçável,Com Vítimas Feridas,Anoitecer,Crescente,Céu Claro,Simples,Reta,Sim,2,0,1,0,1,0,1,1,"-7,2540716","-35,8308372",SPRF-PB,DEL02-PB,UOP01-DEL02-PB


In [ ]:

# Tipos de dados e memória utilizada
df.info(memory_usage="deep")

# Criando o resumo dos tipos de colunas
resumo_tipos = (
    df.dtypes.astype(str)
    .value_counts()
    .rename_axis("tipo")
    .reset_index(name="qtd_colunas")
)

display(resumo_tipos)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72529 entries, 0 to 72528
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   id                      72529 non-null  int64 
 1   data_inversa            72529 non-null  object
 2   dia_semana              72529 non-null  object
 3   horario                 72529 non-null  object
 4   uf                      72529 non-null  object
 5   br                      72529 non-null  int64 
 6   km                      72529 non-null  object
 7   municipio               72529 non-null  object
 8   causa_acidente          72529 non-null  object
 9   tipo_acidente           72529 non-null  object
 10  classificacao_acidente  72528 non-null  object
 11  fase_dia                72529 non-null  object
 12  sentido_via             72529 non-null  object
 13  condicao_metereologica  72529 non-null  object
 14  tipo_pista              72529 non-null  object
 15  tr

,tipo,qtd_colunas
0,object,20
1,int64,10


In [ ]:
# Diagnóstico de valores ausentes
nulos = pd.DataFrame({
    "qtd_nulos": df.isna().sum(),
    "perc_nulos": df.isna().mean() * 100
}).sort_values("perc_nulos", ascending=False)

# Mostra apenas as colunas que possuem pelo menos 1 valor nulo
display(nulos[nulos["qtd_nulos"] > 0])


,qtd_nulos,perc_nulos
uop,38,0.052393
delegacia,22,0.030333
regional,2,0.002758
classificacao_acidente,1,0.001379


In [ ]:
# Diagnóstico e remoção de duplicidades
qtd_duplicadas = df.duplicated().sum()
print("Duplicidades exatas:", qtd_duplicadas)

if qtd_duplicadas > 0:
    df = df.drop_duplicates().copy()
    print("Duplicidades removidas.")
    print("Nova dimensão:", df.shape)


Duplicidades exatas: 0


In [ ]:
# Cardinalidade das variáveis categóricas
categoricas = df.select_dtypes(include="object").columns

cardinalidade = (
    df[categoricas]
    .nunique(dropna=True)
    .sort_values(ascending=False)
    .reset_index()
)

cardinalidade.columns = ["variavel", "qtd_categorias"]
display(cardinalidade.head(30))


,variavel,qtd_categorias
0,latitude,69294
1,longitude,69237
2,km,7655
3,municipio,1844
4,horario,1412
5,tracado_via,605
6,uop,395
7,data_inversa,365
8,delegacia,153
9,causa_acidente,69


In [ ]:
colunas_numericas = [
    "br", "km", "pessoas", "mortos", "feridos",
    "feridos_leves", "feridos_graves", "ilesos", "ignorados", "veiculos"
]

# Converte cada coluna para número de forma segura
for coluna in colunas_numericas:
    if coluna in df.columns:
        df[coluna] = pd.to_numeric(df[coluna], errors='coerce')

# Confere se os tipos mudaram para float64 ou int64
colunas_existentes = [c for c in colunas_numericas if c in df.columns]
print(df[colunas_existentes].dtypes)


br                  int64
km                float64
pessoas             int64
mortos              int64
feridos             int64
feridos_leves       int64
feridos_graves      int64
ilesos              int64
ignorados           int64
veiculos            int64
dtype: object


In [ ]:
# Converte a coluna de data para o formato correto do Python
df["data_inversa"] = pd.to_datetime(df["data_inversa"], errors="coerce")

# Criação de novas variáveis baseadas na data
df["ano"] = df["data_inversa"].dt.year
df["mes"] = df["data_inversa"].dt.month
df["trimestre"] = df["data_inversa"].dt.quarter
df["dia_semana_num"] = df["data_inversa"].dt.dayofweek

# Cria uma coluna que vale 1 se for Sábado (5) ou Domingo (6), e 0 se for dia de semana
df["fim_de_semana"] = df["dia_semana_num"].isin([5, 6]).astype(int)


In [ ]:
# Limpa e extrai apenas a hora do texto
horario_limpo = df["horario"].astype(str).str.strip()
df["hora"] = pd.to_datetime(horario_limpo, format="%H:%M:%S", errors="coerce").dt.hour

# Função organizada com quebras de linha corretas
def classificar_turno(hora):
    if pd.isna(hora):
        return "IGNORADO"
    if 0 <= hora <= 5:
        return "MADRUGADA"
    if 6 <= hora <= 11:
        return "MANHA"
    if 12 <= hora <= 17:
        return "TARDE"
    return "NOITE"

# Aplica a função para criar a nova coluna no DataFrame
df["turno"] = df["hora"].apply(classificar_turno)
